# SBI Diagnostics For gNFW Parameter Subsets

This notebook is based on `sbi_diagnostics_jupyter-Copy1.ipynb`, but it is tuned for diagnostics where the inferred parameter vector is any subset of the 9 gNFW parameters. Edit `SELECTED_PARAM_NAMES` in the configuration cell, then run the notebook top to bottom.

The data-vector input format is the same as before: the notebook accepts combined SBI `.npz` files with `theta`, `theta_columns`, `ell`, and either direct `x_log10_dl`/`x_binned` data or spectra such as `cl_y100`, `cl_stack`, or `cl_mean`.

In [ ]:
from __future__ import annotations

import inspect
import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sbi.inference import NPE
from sbi.utils import BoxUniform

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

## Configuration

Choose the constrained subspace in `SELECTED_PARAM_NAMES`. Parameters that are not selected are not passed to SBI; they are therefore marginalized over through the simulation distribution used to generate `x`.

In [ ]:
CWD = Path.cwd().resolve()
if CWD.name == "SBI_analysis":
    PROJECT_ROOT = CWD.parent
elif (CWD / "SBI_analysis").is_dir():
    PROJECT_ROOT = CWD
elif (CWD.parent / "SBI_analysis").is_dir():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD

SBI_DIR = PROJECT_ROOT / "SBI_analysis"
if str(SBI_DIR) not in sys.path:
    sys.path.insert(0, str(SBI_DIR))

FULL_PARAM_NAMES = [
    "P0_A0",
    "P0_alpha_m",
    "P0_alpha_z",
    "xc_A0",
    "xc_alpha_m",
    "xc_alpha_z",
    "beta_A0",
    "beta_alpha_m",
    "beta_alpha_z",
]

# Default is intentionally fewer than 9 parameters. Edit this list freely.
SELECTED_PARAM_NAMES = ["P0_A0", "P0_alpha_m", "P0_alpha_z", "xc_A0", "beta_A0"]
# SELECTED_PARAM_NAMES = ["P0_A0", "xc_A0", "beta_A0"]
# SELECTED_PARAM_NAMES = ["P0_alpha_m", "P0_alpha_z", "xc_alpha_m", "xc_alpha_z", "beta_alpha_m", "beta_alpha_z"]
# SELECTED_PARAM_NAMES = FULL_PARAM_NAMES.copy()

# Current and older datasets use slightly different names for the same 9 parameters.
PARAM_ALIASES = {
    "P0_A0": ["P0_A0", "P0"],
    "P0_alpha_m": ["P0_alpha_m", "alpha_m_P0"],
    "P0_alpha_z": ["P0_alpha_z", "alpha_z_P0"],
    "xc_A0": ["xc_A0", "xc"],
    "xc_alpha_m": ["xc_alpha_m", "alpha_m_xc"],
    "xc_alpha_z": ["xc_alpha_z", "alpha_z_xc"],
    "beta_A0": ["beta_A0", "beta"],
    "beta_alpha_m": ["beta_alpha_m", "alpha_m_beta"],
    "beta_alpha_z": ["beta_alpha_z", "alpha_z_beta"],
}

SOBOL_PRIOR_BOUNDS = {
    "P0": [1.832524, 34.341221],
    "xc": [0.150011, 0.844503],
    "beta": [3.480627, 5.216611],
    "alpha_m_P0": [0.000312, 0.292251],
    "alpha_m_xc": [-0.099718, 0.099795],
    "alpha_m_beta": [-0.019935, 0.099767],
    "alpha_z_P0": [-1.363457, -0.228839],
    "alpha_z_xc": [0.147393, 1.314474],
    "alpha_z_beta": [0.083808, 0.745884],
}

# Emulator-generated binned dataset. It already contains x_log10_dl with 32 bins.
DATASET_PATH = PROJECT_ROOT / "emulator_tSZ" / "outputs" / "binned_32" / "sbi_dataset_100_000.npz"
METADATA_PATH = DATASET_PATH.with_name(DATASET_PATH.stem + "_metadata.csv")
MANIFEST_PATH = DATASET_PATH.with_name(DATASET_PATH.stem + "_manifest.json")

# Raw 4096-ell dataset option. Uncomment these lines when switching back.
# DATASET_PATH = PROJECT_ROOT / "HPC_output" / "HalfDome" / "ydata_4096" / "sbi_battaglia_y100_4096.npz"
# METADATA_PATH = DATASET_PATH.with_name(DATASET_PATH.stem + "_metadata.csv")
# MANIFEST_PATH = DATASET_PATH.with_name(DATASET_PATH.stem + "_manifest.json")

ELL_MIN = 2
ELL_MAX = 4096

# The emulator dataset is already binned. Enable this only for raw ell grids.
DATAVECTOR_BINNING = {
    "enabled": False,
    "mode": "log",
    "n_bins": 32,
    "ell_min": ELL_MIN,
    "ell_max": ELL_MAX,
    "custom_edges": None,
    "statistic": "mean",
    "weighting": "uniform",
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_SEED = 42

# Training/validation sizes. Keep these moderate for local diagnostics, then increase if needed.
TRAIN_FRACTION = 0.85
MAX_TRAIN_ROWS = 20_000
N_VALIDATION_ROWS = 300

# SBC settings.
NUM_SBC_SIMULATIONS = 200
NUM_SBC_POSTERIOR_SAMPLES = 1000
RANK_NUM_BINS = 20

# Sampling controls. If direct posterior rejection sampling stalls, try POSTERIOR_SAMPLE_WITH = "mcmc".
POSTERIOR_SAMPLE_WITH = None
REJECT_OUTSIDE_PRIOR = True
MAX_SAMPLING_TIME = None
RETURN_PARTIAL_ON_TIMEOUT = False

SAVE_OUTPUTS = True
OUTPUT_DIR = PROJECT_ROOT / "SBI_analysis" / "outputs" / "diagnostics_gNFW_subsets"

print(f"Dataset: {DATASET_PATH}")
print(f"Metadata: {METADATA_PATH if METADATA_PATH.exists() else '<not found>'}")
print(f"Manifest: {MANIFEST_PATH if MANIFEST_PATH.exists() else '<not found>'}")
print(f"Selected parameters ({len(SELECTED_PARAM_NAMES)}): {SELECTED_PARAM_NAMES}")
print(f"Device: {DEVICE}")

## Dataset Helpers

In [ ]:
def as_str_list(values):
    return [str(v) for v in np.asarray(values).tolist()]


def seed_sort_key(name):
    label = name[3:] if name.startswith("cl_") else name
    if label.startswith("y") and label[1:].isdigit():
        return (0, int(label[1:]))
    return (1, label)


def seed_label_from_key(key):
    return key[3:] if key.startswith("cl_") else key


def selected_ell_mask(ell, ell_min=2, ell_max=None):
    ell = np.asarray(ell, dtype=np.float64).reshape(-1)
    mask = ell >= float(ell_min)
    if ell_max is not None:
        mask &= ell <= float(ell_max)
    return mask


def cl_to_log10_dl(cl, ell, floor=1.0e-40):
    cl = np.asarray(cl, dtype=np.float64)
    ell = np.asarray(ell, dtype=np.float64).reshape(-1)
    prefactor = ell * (ell + 1.0) / (2.0 * np.pi)
    dl = cl * prefactor.reshape(1, -1)
    return np.log10(np.clip(dl, floor, None))


def mean_seed_spectra(seed_spectra):
    if not seed_spectra:
        return None
    shapes = {values.shape for values in seed_spectra.values()}
    if len(shapes) != 1:
        raise ValueError(f"Seed spectra have inconsistent shapes: {shapes}")
    if len(seed_spectra) == 1:
        return next(iter(seed_spectra.values()))
    return np.mean(np.stack(list(seed_spectra.values()), axis=0), axis=0)


def npz_scalar_or_none(data, key):
    if key not in data.files:
        return None
    value = np.asarray(data[key])
    if value.shape == ():
        value = value.item()
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    return value


def npz_json_or_none(data, key):
    value = npz_scalar_or_none(data, key)
    if value is None:
        return None
    if isinstance(value, dict):
        return value
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    if isinstance(value, str):
        return json.loads(value)
    return value


def apply_feature_mask(x, ell, mask, label):
    x = np.asarray(x, dtype=np.float32)
    ell = np.asarray(ell, dtype=np.float64).reshape(-1)
    mask = np.asarray(mask, dtype=bool).reshape(-1)
    if x.ndim != 2:
        raise ValueError(f"{label} must be 2D with shape (n_rows, n_ell), got {x.shape}")
    if x.shape[1] == ell.size:
        return x[:, mask]
    if x.shape[1] == int(mask.sum()):
        return x
    raise ValueError(f"{label} has {x.shape[1]} columns, but ell has {ell.size} and selected ell has {int(mask.sum())}")

In [ ]:
def load_sbi_dataset(path: Path, ell_min=2, ell_max=4096):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    with np.load(path, allow_pickle=True) as data:
        keys = set(data.files)
        required = {"theta", "theta_columns", "ell"}
        missing = required - keys
        if missing:
            raise KeyError(f"{path} is missing required keys: {sorted(missing)}")

        theta = np.asarray(data["theta"], dtype=np.float32)
        theta_columns = as_str_list(data["theta_columns"])
        ell_raw = np.asarray(data["ell"], dtype=np.float64).reshape(-1)
        mask = selected_ell_mask(ell_raw, ell_min=ell_min, ell_max=ell_max)
        ell_selected = ell_raw[mask]

        seed_spectra = {}
        target_kind = str(npz_scalar_or_none(data, "target_kind") or "")
        value_kind = None
        pre_binned = False

        if "x_log10_dl" in keys:
            x = apply_feature_mask(data["x_log10_dl"], ell_raw, mask, "x_log10_dl")
            value_kind = "x_log10_dl"
            pre_binned = "binned" in target_kind.lower() or x.shape[1] != ell_raw.size
        elif "x_binned" in keys:
            x = apply_feature_mask(data["x_binned"], ell_raw, mask, "x_binned")
            value_kind = "x_binned"
            pre_binned = True
        elif "x" in keys and any(token in target_kind.lower() for token in ["log10", "binned", "d_ell", "dl"]):
            x = apply_feature_mask(data["x"], ell_raw, mask, "x")
            value_kind = f"x ({target_kind})"
            pre_binned = "binned" in target_kind.lower() or x.shape[1] != ell_raw.size
        else:
            seed_keys = sorted((key for key in keys if key.startswith("cl_y")), key=seed_sort_key)
            for key in seed_keys:
                seed_spectra[seed_label_from_key(key)] = np.asarray(data[key], dtype=np.float64)[:, mask]

            if seed_spectra:
                cl_mean = mean_seed_spectra(seed_spectra)
                value_kind = "mean cl_y* -> log10_D_ell"
            elif "cl_stack" in keys:
                cl_stack = np.asarray(data["cl_stack"], dtype=np.float64)
                if cl_stack.ndim != 3:
                    raise ValueError(f"Expected cl_stack shape (n, n_seeds, n_ell), got {cl_stack.shape}")
                cl_stack = cl_stack[:, :, mask]
                cl_mean = np.mean(cl_stack, axis=1)
                value_kind = "cl_stack mean -> log10_D_ell"
            elif "cl_mean" in keys:
                cl_mean = np.asarray(data["cl_mean"], dtype=np.float64)[:, mask]
                value_kind = "cl_mean -> log10_D_ell"
            else:
                raise KeyError("Could not find x_log10_dl/x_binned/x or cl_y*/cl_stack/cl_mean in dataset")

            x = cl_to_log10_dl(cl_mean, ell_selected).astype(np.float32)

        if theta.shape[0] != x.shape[0]:
            raise ValueError(f"theta rows {theta.shape[0]} and x rows {x.shape[0]} do not match")

        out = {
            "path": path,
            "theta_full": theta,
            "theta_columns": theta_columns,
            "ell": ell_selected.astype(np.float32),
            "ell_raw": ell_raw.astype(np.float32),
            "ell_mask": mask,
            "x": np.ascontiguousarray(x, dtype=np.float32),
            "value_kind": value_kind,
            "pre_binned": bool(pre_binned),
            "target_kind": target_kind,
            "seed_names": list(seed_spectra.keys()),
            "prior_bounds_json": npz_json_or_none(data, "prior_bounds_json"),
        }

        for key in ["prior_low", "prior_high", "bin_counts", "bin_ell_min", "bin_ell_max", "binning_json"]:
            if key in keys:
                if key == "binning_json":
                    out[key] = npz_json_or_none(data, key)
                else:
                    out[key] = np.asarray(data[key])

    return out


dataset = load_sbi_dataset(DATASET_PATH, ell_min=ELL_MIN, ell_max=ELL_MAX)

print("dataset theta columns:", dataset["theta_columns"])
print("theta_full:", dataset["theta_full"].shape)
print("x:", dataset["x"].shape, dataset["value_kind"])
print("ell:", dataset["ell"].shape, float(dataset["ell"][0]), float(dataset["ell"][-1]))
print("pre_binned:", dataset["pre_binned"])

## Select Parameter Subset

This is where the less-than-9 parameter diagnostics are enforced. `theta` is sliced down to `SELECTED_PARAM_NAMES`; `x` remains generated from the full simulation distribution.

In [ ]:
def resolve_param_name(requested_name, available_columns):
    available_columns = list(available_columns)
    aliases = PARAM_ALIASES.get(requested_name, [requested_name])
    for alias in aliases:
        if alias in available_columns:
            return alias
    raise KeyError(f"Parameter {requested_name!r} not found. Tried aliases {aliases}. Available columns: {available_columns}")


def selected_prior_bounds(dataset, selected_names, source_names, source_indices):
    if "prior_low" in dataset and "prior_high" in dataset:
        low_all = np.asarray(dataset["prior_low"], dtype=np.float32).reshape(-1)
        high_all = np.asarray(dataset["prior_high"], dtype=np.float32).reshape(-1)
        if low_all.size == len(dataset["theta_columns"]) and high_all.size == len(dataset["theta_columns"]):
            return low_all[source_indices], high_all[source_indices], "dataset prior_low/prior_high"

    prior_json = dataset.get("prior_bounds_json")
    if isinstance(prior_json, dict):
        bounds = prior_json.get("prior", prior_json)
        low = []
        high = []
        for display_name, source_name in zip(selected_names, source_names):
            candidates = [source_name] + PARAM_ALIASES.get(display_name, [display_name])
            match = next((name for name in candidates if name in bounds), None)
            if match is None:
                break
            lo, hi = bounds[match]
            low.append(float(lo))
            high.append(float(hi))
        if len(low) == len(selected_names):
            return np.asarray(low, dtype=np.float32), np.asarray(high, dtype=np.float32), "dataset prior_bounds_json"

    low = []
    high = []
    for display_name, source_name in zip(selected_names, source_names):
        candidates = [source_name] + PARAM_ALIASES.get(display_name, [display_name])
        match = next((name for name in candidates if name in SOBOL_PRIOR_BOUNDS), None)
        if match is None:
            break
        lo, hi = SOBOL_PRIOR_BOUNDS[match]
        low.append(float(lo))
        high.append(float(hi))
    if len(low) == len(selected_names):
        return np.asarray(low, dtype=np.float32), np.asarray(high, dtype=np.float32), "SOBOL_PRIOR_BOUNDS fallback"

    theta_subset = dataset["theta_full"][:, source_indices]
    return theta_subset.min(axis=0), theta_subset.max(axis=0), "sample min/max fallback"


if not SELECTED_PARAM_NAMES:
    raise ValueError("SELECTED_PARAM_NAMES cannot be empty")
if len(set(SELECTED_PARAM_NAMES)) != len(SELECTED_PARAM_NAMES):
    raise ValueError(f"SELECTED_PARAM_NAMES contains duplicates: {SELECTED_PARAM_NAMES}")

source_param_names = [resolve_param_name(name, dataset["theta_columns"]) for name in SELECTED_PARAM_NAMES]
source_param_indices = [dataset["theta_columns"].index(name) for name in source_param_names]

param_names = list(SELECTED_PARAM_NAMES)
theta_np = np.ascontiguousarray(dataset["theta_full"][:, source_param_indices], dtype=np.float32)
prior_low_np, prior_high_np, prior_source = selected_prior_bounds(
    dataset,
    param_names,
    source_param_names,
    source_param_indices,
)

param_info = pd.DataFrame({
    "parameter": param_names,
    "source_column": source_param_names,
    "prior_low": prior_low_np,
    "prior_high": prior_high_np,
    "theta_min": theta_np.min(axis=0),
    "theta_max": theta_np.max(axis=0),
})

print("selected display names:", param_names)
print("selected source columns:", source_param_names)
print("theta_np:", theta_np.shape)
print("prior source:", prior_source)
display(param_info)

## Optional Data-Vector Binning

For emulator-generated `x_log10_dl` datasets this should normally stay disabled because the input is already binned. Enable it when using a raw 4096-ell dataset.

In [ ]:
def _bin_weights(ell_values, weighting="uniform"):
    ell_values = np.asarray(ell_values, dtype=np.float64)
    weighting = str(weighting).lower()
    if weighting in {"uniform", "none", "flat"}:
        return np.ones_like(ell_values, dtype=np.float64)
    if weighting == "ell":
        return ell_values.astype(np.float64)
    if weighting in {"2ell_plus_1", "modes", "mode_count"}:
        return 2.0 * ell_values + 1.0
    raise ValueError("weighting must be 'uniform', 'ell', or '2ell_plus_1'")


def _make_bin_edges(ell_values, config):
    ell_values = np.asarray(ell_values, dtype=np.float64)
    ell_min = float(config.get("ell_min", np.nanmin(ell_values)))
    ell_max = float(config.get("ell_max", np.nanmax(ell_values)))
    ell_min = max(ell_min, float(np.nanmin(ell_values)))
    ell_max = min(ell_max, float(np.nanmax(ell_values)))
    if not ell_max > ell_min:
        raise ValueError(f"Need ell_max > ell_min for binning, got {ell_min} and {ell_max}")

    mode = str(config.get("mode", "log")).lower()
    if mode == "custom" or config.get("custom_edges") is not None:
        edges = np.asarray(config["custom_edges"], dtype=np.float64)
    else:
        n_bins = int(config.get("n_bins", 32))
        if n_bins < 1:
            raise ValueError("n_bins must be positive")
        if mode == "log":
            if ell_min <= 0:
                raise ValueError("log binning requires ell_min > 0")
            edges = np.geomspace(ell_min, ell_max, n_bins + 1)
        elif mode == "linear":
            edges = np.linspace(ell_min, ell_max, n_bins + 1)
        else:
            raise ValueError("mode must be 'log', 'linear', or 'custom'")

    edges = np.unique(np.asarray(edges, dtype=np.float64))
    if edges.size < 2 or np.any(np.diff(edges) <= 0):
        raise ValueError("Binning edges must be strictly increasing")
    return edges


def bin_datavector(x, ell_values, config):
    x = np.asarray(x, dtype=np.float32)
    ell_values = np.asarray(ell_values, dtype=np.float64).reshape(-1)
    if x.shape[1] != ell_values.size:
        raise ValueError(f"x has {x.shape[1]} columns but ell has {ell_values.size}")

    edges = _make_bin_edges(ell_values, config)
    statistic = str(config.get("statistic", "mean")).lower()
    weighting = config.get("weighting", "uniform")

    binned = []
    centers = []
    counts = []
    ell_min_values = []
    ell_max_values = []

    for i in range(edges.size - 1):
        if i == edges.size - 2:
            in_bin = (ell_values >= edges[i]) & (ell_values <= edges[i + 1])
        else:
            in_bin = (ell_values >= edges[i]) & (ell_values < edges[i + 1])
        if not np.any(in_bin):
            continue

        values = x[:, in_bin]
        ell_bin = ell_values[in_bin]
        if statistic == "mean":
            weights = _bin_weights(ell_bin, weighting=weighting)
            y = np.average(values, axis=1, weights=weights)
        elif statistic == "median":
            y = np.median(values, axis=1)
        else:
            raise ValueError("statistic must be 'mean' or 'median'")

        binned.append(y.astype(np.float32))
        centers.append(np.average(ell_bin, weights=_bin_weights(ell_bin, weighting=weighting)))
        counts.append(int(in_bin.sum()))
        ell_min_values.append(float(ell_bin.min()))
        ell_max_values.append(float(ell_bin.max()))

    if not binned:
        raise ValueError("No non-empty bins were produced")

    return (
        np.stack(binned, axis=1).astype(np.float32),
        np.asarray(centers, dtype=np.float32),
        {
            "enabled": True,
            "edges": edges.astype(np.float32),
            "bin_counts": np.asarray(counts, dtype=np.int32),
            "bin_ell_min": np.asarray(ell_min_values, dtype=np.float32),
            "bin_ell_max": np.asarray(ell_max_values, dtype=np.float32),
            "config": dict(config),
        },
    )


x_np = np.ascontiguousarray(dataset["x"], dtype=np.float32)
ell_np = np.asarray(dataset["ell"], dtype=np.float32)

if DATAVECTOR_BINNING.get("enabled", False):
    if dataset["pre_binned"]:
        print("WARNING: input appears pre-binned, but DATAVECTOR_BINNING is enabled. Binning active x again.")
    x_np, ell_np, BINNING_SUMMARY = bin_datavector(x_np, ell_np, DATAVECTOR_BINNING)
else:
    BINNING_SUMMARY = {
        "enabled": False,
        "config": dict(DATAVECTOR_BINNING),
        "bin_counts": np.asarray(dataset.get("bin_counts", np.ones_like(ell_np, dtype=np.int32))),
        "bin_ell_min": np.asarray(dataset.get("bin_ell_min", ell_np), dtype=np.float32),
        "bin_ell_max": np.asarray(dataset.get("bin_ell_max", ell_np), dtype=np.float32),
    }

print("active theta_np:", theta_np.shape)
print("active x_np:", x_np.shape)
print("active ell_np:", ell_np.shape, float(ell_np[0]), float(ell_np[-1]))
print("binning enabled:", BINNING_SUMMARY["enabled"])

## Train/Validation Split And Prior

In [ ]:
def to_tensor(x, device=None):
    if hasattr(x, "detach"):
        out = x.detach().float()
    else:
        out = torch.as_tensor(x, dtype=torch.float32)
    if device is not None:
        out = out.to(device)
    return out


def split_train_validation(theta_np, x_np, train_fraction=0.85, max_train_rows=None, n_validation_rows=300, seed=0):
    theta = to_tensor(theta_np)
    x = to_tensor(x_np)
    n_total = theta.shape[0]
    if n_total < 3:
        raise ValueError("Need at least 3 simulations for a train/validation split")

    rng = torch.Generator().manual_seed(int(seed))
    indices = torch.randperm(n_total, generator=rng)

    n_train_candidate = int(math.floor(n_total * float(train_fraction)))
    if max_train_rows is not None:
        n_train_candidate = min(n_train_candidate, int(max_train_rows))
    n_train = max(1, min(n_train_candidate, n_total - 1))

    n_val_available = n_total - n_train
    n_val = min(int(n_validation_rows), n_val_available)
    if n_val < 1:
        n_train = n_total - 1
        n_val = 1

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]

    return {
        "theta_train": theta[train_idx],
        "x_train": x[train_idx],
        "theta_val": theta[val_idx],
        "x_val": x[val_idx],
        "train_idx": train_idx,
        "val_idx": val_idx,
    }


prior_low_cpu = torch.from_numpy(np.asarray(prior_low_np, dtype=np.float32))
prior_high_cpu = torch.from_numpy(np.asarray(prior_high_np, dtype=np.float32))
prior_low = prior_low_cpu.to(DEVICE)
prior_high = prior_high_cpu.to(DEVICE)

try:
    prior = BoxUniform(low=prior_low, high=prior_high, device=DEVICE)
except TypeError:
    prior = BoxUniform(low=prior_low, high=prior_high)

split = split_train_validation(
    theta_np,
    x_np,
    train_fraction=TRAIN_FRACTION,
    max_train_rows=MAX_TRAIN_ROWS,
    n_validation_rows=N_VALIDATION_ROWS,
    seed=RANDOM_SEED,
)

prepared = {
    "theta": theta_np,
    "x": x_np,
    "ell": ell_np,
    "param_names": param_names,
    "source_param_names": source_param_names,
    "prior": prior,
    "prior_low": prior_low_cpu,
    "prior_high": prior_high_cpu,
    "split": split,
    "binning": BINNING_SUMMARY,
}

print("theta_train:", split["theta_train"].shape)
print("x_train:", split["x_train"].shape)
print("theta_val:", split["theta_val"].shape)
print("x_val:", split["x_val"].shape)

## Train NPE Posterior

This posterior is trained only in the selected parameter space. The unselected parameters are nuisance directions marginalized over by the simulations in the dataset.

In [ ]:
torch.manual_seed(RANDOM_SEED)

inference = NPE(prior=prepared["prior"])

theta_train = split["theta_train"].to(DEVICE)
x_train = split["x_train"].to(DEVICE)

density_estimator = inference.append_simulations(theta_train, x_train).train()

build_kwargs = {}
if POSTERIOR_SAMPLE_WITH is not None:
    build_kwargs["sample_with"] = POSTERIOR_SAMPLE_WITH

posterior = inference.build_posterior(density_estimator, **build_kwargs)
posterior

## Posterior Sampling Helpers

These helpers sample validation posteriors with version-aware arguments. If sampling stalls because rejection outside the prior is too aggressive, set `REJECT_OUTSIDE_PRIOR = False` for a quick diagnostic or use `POSTERIOR_SAMPLE_WITH = "mcmc"` before retraining the posterior cell.

In [ ]:
def posterior_sample_signature(posterior):
    try:
        return inspect.signature(posterior.sample)
    except Exception:
        return None


def sample_posterior_for_x(
    posterior,
    x_i,
    num_samples,
    reject_outside_prior=REJECT_OUTSIDE_PRIOR,
    max_sampling_time=MAX_SAMPLING_TIME,
    return_partial_on_timeout=RETURN_PARTIAL_ON_TIMEOUT,
    show_progress_bars=False,
):
    sig = posterior_sample_signature(posterior)
    allowed = set(sig.parameters) if sig is not None else set()

    kwargs = {"x": x_i.to(DEVICE) if hasattr(x_i, "to") else to_tensor(x_i, DEVICE)}
    if "show_progress_bars" in allowed:
        kwargs["show_progress_bars"] = show_progress_bars
    if "reject_outside_prior" in allowed:
        kwargs["reject_outside_prior"] = reject_outside_prior
    if max_sampling_time is not None and "max_sampling_time" in allowed:
        kwargs["max_sampling_time"] = max_sampling_time
    if "return_partial_on_timeout" in allowed:
        kwargs["return_partial_on_timeout"] = return_partial_on_timeout

    return posterior.sample((int(num_samples),), **kwargs).detach().cpu()


def sample_validation_posteriors(posterior, theta_val, x_val, num_simulations, num_posterior_samples):
    theta_val = to_tensor(theta_val)
    x_val = to_tensor(x_val)
    n = min(int(num_simulations), theta_val.shape[0])
    samples = []
    ranks = []

    for i in range(n):
        samples_i = sample_posterior_for_x(
            posterior,
            x_val[i],
            num_posterior_samples,
            show_progress_bars=False,
        )
        samples.append(samples_i)
        ranks_i = (samples_i < theta_val[i].reshape(1, -1)).sum(dim=0)
        ranks.append(ranks_i)
        if (i + 1) % 25 == 0 or i + 1 == n:
            print(f"sampled validation posterior {i + 1}/{n}")

    return torch.stack(samples, dim=0), torch.stack(ranks, dim=0), theta_val[:n]

## Run SBC Sampling

`validation_samples` has shape `(n_sbc, num_posterior_samples, n_selected_params)`. `ranks` has shape `(n_sbc, n_selected_params)`.

In [ ]:
validation_samples, ranks, theta_sbc = sample_validation_posteriors(
    posterior,
    split["theta_val"],
    split["x_val"],
    num_simulations=NUM_SBC_SIMULATIONS,
    num_posterior_samples=NUM_SBC_POSTERIOR_SAMPLES,
)

print("validation_samples:", tuple(validation_samples.shape))
print("ranks:", tuple(ranks.shape))
print("theta_sbc:", tuple(theta_sbc.shape))

## SBC Summary Table

In [ ]:
def sbc_summary_table(ranks, posterior_samples, theta_true, labels, num_posterior_samples, prior_low, prior_high):
    ranks_np = to_tensor(ranks).numpy()
    samples = to_tensor(posterior_samples)
    theta_true = to_tensor(theta_true)
    prior_low = to_tensor(prior_low).reshape(-1)
    prior_high = to_tensor(prior_high).reshape(-1)
    prior_delta = prior_high - prior_low

    posterior_mean = samples.mean(dim=1)
    posterior_std = samples.std(dim=1)
    error = posterior_mean - theta_true

    expected_rank_mean = num_posterior_samples / 2.0
    expected_rank_std = math.sqrt(num_posterior_samples * (num_posterior_samples + 2.0) / 12.0)

    rows = []
    for j, name in enumerate(labels):
        rank_j = ranks_np[:, j]
        mean_error = error[:, j].mean().item()
        rows.append({
            "parameter": name,
            "rank_mean": float(np.mean(rank_j)),
            "rank_mean_z": float((np.mean(rank_j) - expected_rank_mean) / (expected_rank_std / math.sqrt(rank_j.size))),
            "rank_std": float(np.std(rank_j, ddof=1)),
            "expected_rank_std": float(expected_rank_std),
            "posterior_mean_bias": mean_error,
            "posterior_mean_bias / prior_delta": float(mean_error / prior_delta[j].item()),
            "mean_abs_error / prior_delta": float((torch.abs(error[:, j]) / prior_delta[j]).mean().item()),
            "mean_posterior_std / prior_delta": float((posterior_std[:, j] / prior_delta[j]).mean().item()),
        })
    return pd.DataFrame(rows)


sbc_summary = sbc_summary_table(
    ranks,
    validation_samples,
    theta_sbc,
    param_names,
    NUM_SBC_POSTERIOR_SAMPLES,
    prior_low_cpu,
    prior_high_cpu,
)

display(
    sbc_summary.style.format({
        "rank_mean": "{:.1f}",
        "rank_mean_z": "{:+.2f}",
        "rank_std": "{:.1f}",
        "expected_rank_std": "{:.1f}",
        "posterior_mean_bias": "{:+.4g}",
        "posterior_mean_bias / prior_delta": "{:+.3g}",
        "mean_abs_error / prior_delta": "{:.3g}",
        "mean_posterior_std / prior_delta": "{:.3g}",
    }).background_gradient(subset=["rank_mean_z"], cmap="coolwarm", vmin=-3, vmax=3)
)

## Cumulative Rank Plot

For calibrated posteriors, the rank CDF should follow the black diagonal within sampling fluctuations.

In [ ]:
def plot_cumulative_rank(ranks, labels, num_posterior_samples, alpha=0.05, figsize=(8, 6)):
    ranks_np = to_tensor(ranks).numpy()
    n_sbc, n_params = ranks_np.shape
    fig, ax = plt.subplots(figsize=figsize)

    grid = np.linspace(0.0, 1.0, 200)
    eps = math.sqrt(math.log(2.0 / alpha) / (2.0 * n_sbc))
    ax.fill_between(grid, np.maximum(0, grid - eps), np.minimum(1, grid + eps), color="0.85", label=f"{int((1-alpha)*100)}% DKW band")
    ax.plot([0, 1], [0, 1], color="black", lw=1.2, ls="--", label="ideal")

    for j, name in enumerate(labels):
        u = (ranks_np[:, j] + 0.5) / (num_posterior_samples + 1.0)
        u = np.sort(u)
        ecdf = np.arange(1, n_sbc + 1) / n_sbc
        ax.step(u, ecdf, where="post", lw=1.4, label=name)

    ax.set_xlabel("normalized rank")
    ax.set_ylabel("empirical CDF")
    ax.set_title("Cumulative SBC rank plot")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    return fig, ax


fig, ax = plot_cumulative_rank(ranks, param_names, NUM_SBC_POSTERIOR_SAMPLES)
plt.show()

## Separate Rank Plots

Each panel is one selected parameter. A flat histogram is the target for calibrated marginal posteriors.

In [ ]:
def plot_rank_histograms(ranks, labels, num_posterior_samples, num_bins=20, max_cols=3):
    ranks_np = to_tensor(ranks).numpy()
    n_sbc, n_params = ranks_np.shape
    ncols = min(max_cols, n_params)
    nrows = int(math.ceil(n_params / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.2 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    bins = np.linspace(0, num_posterior_samples + 1, num_bins + 1)
    expected = n_sbc / num_bins
    sigma = math.sqrt(n_sbc * (1.0 / num_bins) * (1.0 - 1.0 / num_bins))

    for j, name in enumerate(labels):
        ax = axes_flat[j]
        ax.fill_between([0, num_posterior_samples + 1], expected - 2 * sigma, expected + 2 * sigma, color="0.85", alpha=0.45, label="approx. 2 sigma", zorder=0)
        ax.hist(ranks_np[:, j], bins=bins, color="#4C72B0", alpha=0.75, edgecolor="white", zorder=2)
        ax.axhline(expected, color="black", ls="--", lw=1.2, label="uniform mean", zorder=3)
        ax.set_title(name)
        ax.set_xlabel("rank")
        ax.set_ylabel("count")
        ax.set_xlim(0, num_posterior_samples + 1)

    for ax in axes_flat[n_params:]:
        ax.axis("off")

    handles, labels_legend = axes_flat[0].get_legend_handles_labels()
    fig.legend(handles, labels_legend, loc="upper right", frameon=False)
    fig.suptitle("SBC rank histograms by parameter", y=1.02)
    fig.tight_layout()
    return fig, axes


fig, axes = plot_rank_histograms(ranks, param_names, NUM_SBC_POSTERIOR_SAMPLES, num_bins=RANK_NUM_BINS)
plt.show()

## True Vs Posterior Mean

Each point is one validation simulation. The black line is the ideal `posterior mean = true value` relation.

In [ ]:
def plot_true_vs_posterior_mean(posterior_samples, theta_true, labels, max_cols=3):
    samples = to_tensor(posterior_samples)
    theta_true = to_tensor(theta_true)
    posterior_mean = samples.mean(dim=1)

    n_params = theta_true.shape[1]
    ncols = min(max_cols, n_params)
    nrows = int(math.ceil(n_params / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.4 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    rows = []
    for j, name in enumerate(labels):
        ax = axes_flat[j]
        x = theta_true[:, j].cpu().numpy()
        y = posterior_mean[:, j].cpu().numpy()
        lo = float(min(np.min(x), np.min(y)))
        hi = float(max(np.max(x), np.max(y)))
        pad = 0.04 * (hi - lo) if hi > lo else 1.0
        lo -= pad
        hi += pad

        ax.scatter(x, y, s=16, alpha=0.65, color="#55A868", edgecolor="none")
        ax.plot([lo, hi], [lo, hi], color="black", ls="--", lw=1.2)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_title(name)
        ax.set_xlabel("true")
        ax.set_ylabel("posterior mean")

        error = y - x
        corr = np.corrcoef(x, y)[0, 1] if x.size > 1 and np.std(x) > 0 and np.std(y) > 0 else np.nan
        rows.append({
            "parameter": name,
            "bias": float(np.mean(error)),
            "mae": float(np.mean(np.abs(error))),
            "rmse": float(np.sqrt(np.mean(error ** 2))),
            "corr_true_mean": float(corr),
        })

    for ax in axes_flat[n_params:]:
        ax.axis("off")

    fig.suptitle("True value vs posterior mean", y=1.02)
    fig.tight_layout()
    return fig, axes, pd.DataFrame(rows)


fig, axes, true_vs_mean_df = plot_true_vs_posterior_mean(validation_samples, theta_sbc, param_names)
plt.show()
display(true_vs_mean_df)

## Rank Uniformity Table

A compact numerical check of the same rank uniformity shown in the CDF and histogram plots.

In [ ]:
def rank_uniformity_table(ranks, labels, num_posterior_samples):
    ranks_np = to_tensor(ranks).numpy()
    rows = []
    try:
        from scipy import stats
    except Exception:
        stats = None

    for j, name in enumerate(labels):
        u = (ranks_np[:, j] + 0.5) / (num_posterior_samples + 1.0)
        row = {
            "parameter": name,
            "n_sbc": int(u.size),
            "mean_rank_fraction": float(np.mean(u)),
            "std_rank_fraction": float(np.std(u, ddof=1)),
            "min_rank_fraction": float(np.min(u)),
            "max_rank_fraction": float(np.max(u)),
        }
        if stats is not None:
            ks = stats.kstest(u, "uniform")
            row["ks_statistic"] = float(ks.statistic)
            row["ks_pvalue"] = float(ks.pvalue)
        rows.append(row)
    return pd.DataFrame(rows)


rank_uniformity_df = rank_uniformity_table(ranks, param_names, NUM_SBC_POSTERIOR_SAMPLES)
display(
    rank_uniformity_df.style.format({
        "mean_rank_fraction": "{:.3f}",
        "std_rank_fraction": "{:.3f}",
        "min_rank_fraction": "{:.3f}",
        "max_rank_fraction": "{:.3f}",
        "ks_statistic": "{:.3f}",
        "ks_pvalue": "{:.3g}",
    })
)

## Save Diagnostics

In [ ]:
def jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    return value


if SAVE_OUTPUTS:
    subset_tag = "_".join(param_names)
    save_dir = OUTPUT_DIR / subset_tag
    save_dir.mkdir(parents=True, exist_ok=True)

    np.save(save_dir / "sbc_ranks.npy", ranks.detach().cpu().numpy())
    np.save(save_dir / "validation_posterior_samples.npy", validation_samples.detach().cpu().numpy())
    np.save(save_dir / "theta_sbc_true.npy", theta_sbc.detach().cpu().numpy())
    sbc_summary.to_csv(save_dir / "sbc_summary.csv", index=False)
    rank_uniformity_df.to_csv(save_dir / "rank_uniformity.csv", index=False)
    true_vs_mean_df.to_csv(save_dir / "true_vs_posterior_mean.csv", index=False)

    summary = {
        "dataset_path": DATASET_PATH,
        "selected_param_names": param_names,
        "source_param_names": source_param_names,
        "theta_shape": theta_np.shape,
        "x_shape": x_np.shape,
        "ell_shape": ell_np.shape,
        "prior_source": prior_source,
        "train_rows": int(split["theta_train"].shape[0]),
        "validation_rows": int(split["theta_val"].shape[0]),
        "num_sbc_simulations": int(theta_sbc.shape[0]),
        "num_sbc_posterior_samples": int(NUM_SBC_POSTERIOR_SAMPLES),
        "binning": BINNING_SUMMARY,
        "posterior_sample_with": POSTERIOR_SAMPLE_WITH,
        "reject_outside_prior": REJECT_OUTSIDE_PRIOR,
    }
    with open(save_dir / "diagnostic_summary.json", "w", encoding="utf-8") as f:
        json.dump(jsonable(summary), f, indent=2)

    print(f"Saved diagnostics to {save_dir}")